In [216]:
import sympy as sp
import portion as P
import math

In [217]:
K=-2
main_interval = (-100,100)
EPS = 0.0001

In [218]:
x_sym = sp.Symbol("x")
k_sym = sp.Symbol("k")

a_sym = sp.Symbol("a")
b_sym = sp.Symbol("b")
c_sym = sp.Symbol("c")

In [219]:
a = 9
b = 1
c = 3

In [220]:
ABC = sorted((a,b,c))
A = ABC[0]
B = ABC[1]
C = ABC[2]
interval = P.closed(-P.inf, A) | P.closed(B,C)
interval -= P.closed(a,a)
interval -= P.closed(b,b)
interval -= P.closed(c,c)
interval

(-inf,1) | (3,9)

необходимое условие \
$k \in (-\infty;1)\cup(3;9)$ \
\
достаточное условие \
$k \in (-\infty;1)\cup(1;3)\cup(3;9)\cup(9;+\infty)$

In [221]:
def func_simple(x: float, k: float) -> float:
    return (k-a)*x + math.e**((k-b)*(k-c)*x)

In [222]:
func_with_k_abc = (k_sym-a_sym)*x_sym + sp.E**((k_sym-b_sym)*(k_sym-c_sym)*x_sym)

In [223]:
func_with_k = func_with_k_abc.subs({
    "a": a,
    "b": b,
    "c": c,
})
func_with_k

x*(k - 9) + exp(x*(k - 3)*(k - 1))

In [224]:
func_with_k_abc_d1 = sp.diff(func_with_k_abc, x_sym)
func_with_k_abc_d1

-a + k + (-b + k)*(-c + k)*exp(x*(-b + k)*(-c + k))

In [225]:
func_with_k_abc_d2 = sp.diff(func_with_k_abc_d1, x_sym)
func_with_k_abc_d2

(-b + k)**2*(-c + k)**2*exp(x*(-b + k)*(-c + k))

In [226]:
def get_global_min(
    k: float,
    a: float = a,
    b: float = b,
    c: float = c,
    verbose: bool = False,
) -> float | None:
    if verbose:
        print(func_with_k_abc_d1, "- f(x)")
    func_d1 = func_with_k_abc_d1.subs({
        "k": k,
        "a": a,
        "b": b,
        "c": c,
    })
    func_d2 = func_with_k_abc_d2.subs({
        "k": k,
        "a": a,
        "b": b,
        "c": c,
    })
    if verbose:
        print("f'(x) - ", func_d1)
        print("f''(x) - ", func_d2)
    solvings = sp.solveset(func_d1, x_sym, domain=sp.S.Reals)
    if not solvings:
        if verbose:
            print("нет решений f'(x)=0")
        return None
    solving = list(solvings)[0]
    if verbose:
        print(f"решение f'(x)=0, x={solving.evalf(5)}")
    result_d2 = func_d2.subs({"x": solving})
    if verbose:
        print("значения f''(x) при подставлении решения f'(x)=0", result_d2.evalf(5))
    if result_d2 <= 0:
        if verbose:
            print(f"значение f''(x) = {result_d2} и т.к. оно <=0, глобального минимума нет")
        return None
    result_simple = func_with_k.subs({
        "k": k,
        "x": solving,
    })
    pm_eps = 0.001
    result_m1 = func_with_k.subs({
        "k": k,
        "x": solving - pm_eps,
    })
    result_p1 = func_with_k.subs({
        "k": k,
        "x": solving + pm_eps,
    })
    if verbose:
        print(f"значение f''(x)={result_d2.evalf(5)}, а значит точка {solving.evalf(5)} - глобавльный минимум")
        print(f"f({solving.evalf(5)})={result_simple.evalf(5)}")
        print(f"f({(solving-1).evalf(5)})={result_m1.evalf(10)}, при x-{pm_eps}")
        print(f"f({(solving+1).evalf(5)})={result_p1.evalf(10)}, при x+{pm_eps}")
    return solving

In [227]:
result = get_global_min(K, verbose=True)
x_a = result.evalf(50)
print(f"результат: x={result.evalf(5)}, f({result.evalf(5)})={func_simple(result.evalf(30),K):.5f}")
print(f"x точный = {result}")


-a + k + (-b + k)*(-c + k)*exp(x*(-b + k)*(-c + k)) - f(x)
f'(x) -  15*exp(15*x) - 11
f''(x) -  225*exp(15*x)
решение f'(x)=0, x=-0.020677
значения f''(x) при подставлении решения f'(x)=0 165.00
значение f''(x)=165.00, а значит точка -0.020677 - глобавльный минимум
f(-0.020677)=0.96078
f(-1.0207)=0.9608623698, при x-0.001
f(0.97932)=0.9608631948, при x+0.001
результат: x=-0.020677, f(-0.020677)=0.96078
x точный = log(11/15)/15


In [228]:
def dichotomy_method(a, b, k, epsilon):
    delta = epsilon / 2.1
    iterations = 0
    while (b - a) / 2 > epsilon:
        iterations += 1
        x1 = (a + b) / 2 - delta
        x2 = (a + b) / 2 + delta
        if func_simple(x1, k) < func_simple(x2, k):
            b = x2
        else:
            a = x1
    x_opt = (a + b) / 2
    return x_opt, func_simple(x_opt, k), iterations

In [229]:
x_d, f_d, iter_d = dichotomy_method(*main_interval, K, EPS)
print(f"x = {x_d:.5f}, f(x) = {f_d:.5f}, итераций: {iter_d}")

x = -0.02065, f(x) = 0.96078, итераций: 21


In [230]:
def golden_section_method(a, b, k, epsilon):
    phi = (math.sqrt(5) - 1) / 2
    x1 = b - phi * (b - a)
    x2 = a + phi * (b - a)
    f1, f2 = func_simple(x1, k), func_simple(x2, k)
    iterations = 0
    while (b - a) / 2 > epsilon:
        iterations += 1
        if f1 < f2:
            b = x2
            x2, f2 = x1, f1
            x1 = b - phi * (b - a)
            f1 = func_simple(x1, k)
        else:
            a = x1
            x1, f1 = x2, f2
            x2 = a + phi * (b - a)
            f2 = func_simple(x2, k)
    x_opt = (a + b) / 2
    return x_opt, func_simple(x_opt, k), iterations

In [231]:
x_g, f_g, iter_g = golden_section_method(*main_interval, K, EPS)
print(f"x = {x_g:.5f}, f(x) = {f_g:.5f}, итераций: {iter_g}")

x = -0.02069, f(x) = 0.96078, итераций: 29


In [232]:
def generate_fibonacci_array(n):
    fib = [1, 1]
    for i in range(2, n + 2):
        fib.append(fib[i-1] + fib[i-2])
    return fib

In [233]:
def fibonacci_method(a, b, n, k, epsilon):
    fib_array = generate_fibonacci_array(n)

    x1 = a + (fib_array[n-2] / fib_array[n]) * (b - a)
    x2 = a + (fib_array[n-1] / fib_array[n]) * (b - a)
    f1, f2 = func_simple(x1, k), func_simple(x2, k)
    for it in range(1, n - 1):
        if f1 < f2:
            b = x2
            x2, f2 = x1, f1
            x1 = a + (fib_array[n-it-2] / fib_array[n-it]) * (b - a)
            f1 = func_simple(x1, k)
        else:
            a = x1
            x1, f1 = x2, f2
            x2 = a + (fib_array[n-it-1] / fib_array[n-it]) * (b - a)
            f2 = func_simple(x2, k)
    x2 = x1 + epsilon
    if func_simple(x1, k) < func_simple(x2, k):
        b = x2
    else:
        a = x1
    x_opt = (a + b) / 2
    return x_opt, func_simple(x_opt, k), n

In [234]:
x_f, f_f, iter_f = fibonacci_method(*main_interval, 100000, K, EPS)
print(f"x = {x_f:.5f}, f(x) = {f_f:.5f}, итераций: {iter_f}")

x = -0.02063, f(x) = 0.96078, итераций: 100000


In [235]:
y_d = func_simple(x_d, K)
y_g = func_simple(x_g, K)
y_f = func_simple(x_f, K)
y_a = func_simple(x_a, K)
y_d_d = func_with_k_abc_d1.subs({"a": a,"b": b,"c": c, "k": K, "x": x_d})
y_g_d = func_with_k_abc_d1.subs({"a": a,"b": b,"c": c, "k": K, "x": x_g})
y_f_d = func_with_k_abc_d1.subs({"a": a,"b": b,"c": c, "k": K, "x": x_f})
y_a_d = func_with_k_abc_d1.subs({"a": a,"b": b,"c": c, "k": K, "x": x_a})

print("сравниваем алгоритмы")
print(f"f({x_d:.10f})={y_d:.10f}, f'({x_d:.10f})={y_d_d:.10f} - решение через метод дихотомии")
print(f"f({x_g:.10f})={y_g:.10f}, f'({x_g:.10f})={y_g_d:.10f} - решение через золотое сечение")
print(f"f({x_f:.10f})={y_f:.10f}, f'({x_f:.10f})={y_f_d:.10f} - решение через Фибоначи")
print(f"f({x_a:.10f})={y_a:.10f}, f'({x_a:.10f})={y_a_d:.10f} - аналитическое решение")

сравниваем алгоритмы
f(-0.0206470391)=0.9607803548, f'(-0.0206470391)=0.0049438675 - решение через метод дихотомии
f(-0.0206890779)=0.9607802928, f'(-0.0206890779)=-0.0019934571 - решение через золотое сечение
f(-0.0206269951)=0.9607804871, f'(-0.0206269951)=0.0082531105 - решение через Фибоначи
f(-0.0206769952)=0.9607802808, f'(-0.0206769952)=0.0000000000 - аналитическое решение
